In [89]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [90]:
import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import RandomOverSampler
import warnings
warnings.filterwarnings('ignore')


In [91]:
# Load the dataset with 'latin1' encoding
df = pd.read_csv('/content/drive/My Drive/Sem-6/ML/Lab/ML_Lab_2/benign_vs_defacement_urls.csv', on_bad_lines='skip',quoting=3,engine='python', encoding='latin1')

print("Original dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())

# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

# Remove rows with missing values
df = df.dropna()
print("\nDataset shape after removing missing values:", df.shape)

# Simplify labels: benign -> good, defacement -> bad
df['label'] = df['type'].map({'benign': 'good', 'defacement': 'bad'})

print("\nLabel distribution:")
print(df['label'].value_counts())

Original dataset shape: (524003, 2)

First few rows:
                                                 url    type
0                mp3raid.com/music/krizz_kaliko.html  benign
1                    bopsecrets.org/rexroth/cr/1.htm  benign
2  http://buzzfil.net/m/show-art/ils-etaient-loin...  benign
3      espn.go.com/nba/player/_/id/3457/brandon-rush  benign
4     yourbittorrent.com/?q=anthony-hamilton-soulife  benign

Missing values:
url      8
type    61
dtype: int64

Dataset shape after removing missing values: (523934, 2)

Label distribution:
label
good    427883
bad      96051
Name: count, dtype: int64


In [92]:
def extract_url_features(url):
    """Extract all features from a URL"""
    features = {}

    # 1. Length of full URL
    features['url_length'] = len(url)

    # 2. Hostname length
    try:
        parsed = urlparse(url)
        hostname = parsed.netloc
        features['hostname_length'] = len(hostname)
    except:
        features['hostname_length'] = 0

    # 3. Symbol counting
    features['count_dot'] = url.count('.')
    features['count_dash'] = url.count('-')
    features['count_at'] = url.count('@')
    features['count_question'] = url.count('?')
    features['count_percent'] = url.count('%')
    features['count_equal'] = url.count('=')

    # 4. Digit count
    features['digit_count'] = sum(c.isdigit() for c in url)

    # 5. Special patterns
    # Check if URL uses IP address
    ip_pattern = r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}'
    features['has_ip'] = 1 if re.search(ip_pattern, url) else 0

    # Check for shortened URL services
    shorteners = ['bit.ly', 'goo.gl', 'tinyurl.com', 't.co', 'ow.ly', 'is.gd']
    features['is_shortened'] = 1 if any(short in url.lower() for short in shorteners) else 0

    # Count slashes (directories)
    features['slash_count'] = url.count('/')

    return features

# Test the function
test_url = "https://www.example.com/page?id=123"
print("Testing feature extraction:")
print(extract_url_features(test_url))

categorical_cols = ['has_ip', 'is_shortened']


Testing feature extraction:
{'url_length': 35, 'hostname_length': 15, 'count_dot': 2, 'count_dash': 0, 'count_at': 0, 'count_question': 1, 'count_percent': 0, 'count_equal': 1, 'digit_count': 3, 'has_ip': 0, 'is_shortened': 0, 'slash_count': 3}


In [93]:
# Reset index of df AFTER dropna in the previous step, to ensure a contiguous index.
# This is crucial for proper alignment when concatenating with features_df.
df = df.reset_index(drop=True)

# Extract features for all URLs
print("Extracting features from all URLs...")
feature_list = []

for url in df['url']:
    features = extract_url_features(url)
    feature_list.append(features)

# Create feature dataframe
features_df = pd.DataFrame(feature_list)

# Combine with labels
# Since df and features_df now both have contiguous 0-based indices, concatenation will work correctly
df_with_features = pd.concat([df[['url', 'label']], features_df], axis=1)

print("\nDataset with extracted features:")
print(df_with_features.head())
print("\nFeature columns:", features_df.columns.tolist())
print("\nDataset shape:", df_with_features.shape)

Extracting features from all URLs...

Dataset with extracted features:
                                                 url label  url_length  \
0                mp3raid.com/music/krizz_kaliko.html  good          35   
1                    bopsecrets.org/rexroth/cr/1.htm  good          31   
2  http://buzzfil.net/m/show-art/ils-etaient-loin...  good         118   
3      espn.go.com/nba/player/_/id/3457/brandon-rush  good          45   
4     yourbittorrent.com/?q=anthony-hamilton-soulife  good          46   

   hostname_length  count_dot  count_dash  count_at  count_question  \
0                0          2           0         0               0   
1                0          2           0         0               0   
2               11          2          16         0               0   
3                0          2           1         0               0   
4                0          1           2         0               1   

   count_percent  count_equal  digit_count  has_ip  is_sh

In [94]:
# Prepare features (X) and target (y)
X = features_df
y = df_with_features['label']

print("Original class distribution:")
print(y.value_counts())

# Apply resampling to balance the classes
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X, y)

print("\nResampled class distribution:")
print(pd.Series(y_resampled).value_counts())
print("\nResampled dataset shape:", X_resampled.shape)


Original class distribution:
label
good    427883
bad      96051
Name: count, dtype: int64

Resampled class distribution:
label
good    427883
bad     427883
Name: count, dtype: int64

Resampled dataset shape: (855766, 12)


In [95]:
X_small, X_unused, y_small, y_unused = train_test_split(
    X_resampled, y_resampled,
    train_size=5000,
    stratify=y_resampled,
    random_state=42
)

# Now we split our small 5000 dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(X_small, y_small, test_size=0.2, random_state=42)

# Convert to numpy for our functions
X_train_arr = np.array(X_train)
y_train_arr = np.array(y_train)
X_test_arr = np.array(X_test)

print(f"Final Training Size: {len(X_train_arr)} rows")
print(f"Final Test Size: {len(X_test_arr)} rows")

Final Training Size: 4000 rows
Final Test Size: 1000 rows


In [96]:
def calculate_entropy(y):
    if len(y) == 0:
        return 0

    values, counts = np.unique(y, return_counts=True)

    entropy = 0
    total = len(y)

    for count in counts:
        prob = count / total
        # Formula: -p * log2(p)
        entropy = entropy - (prob * np.log2(prob))

    return entropy

In [97]:
def calculate_numerical_gain(column_data, y, threshold):
    parent_entropy = calculate_entropy(y)

    # Split the data
    left_mask = column_data <= threshold
    right_mask = column_data > threshold

    left_y = y[left_mask]
    right_y = y[right_mask]

    # If split doesn't divide data, gain is 0
    if len(left_y) == 0 or len(right_y) == 0:
        return 0

    # Weighted average of children entropy
    n = len(y)
    n_left = len(left_y)
    n_right = len(right_y)

    e_left = calculate_entropy(left_y)
    e_right = calculate_entropy(right_y)

    child_entropy = (n_left / n) * e_left + (n_right / n) * e_right

    info_gain = parent_entropy - child_entropy
    return info_gain

In [98]:
def calculate_categorical_gain(column_data, y, category_value):
    parent_entropy = calculate_entropy(y)

    # Split the data (Equality check)
    match_mask = column_data == category_value
    not_match_mask = column_data != category_value

    match_y = y[match_mask]
    not_match_y = y[not_match_mask]

    if len(match_y) == 0 or len(not_match_y) == 0:
        return 0

    n = len(y)
    n_match = len(match_y)
    n_not_match = len(not_match_y)

    e_match = calculate_entropy(match_y)
    e_not_match = calculate_entropy(not_match_y)

    child_entropy = (n_match / n) * e_match + (n_not_match / n) * e_not_match

    info_gain = parent_entropy - child_entropy
    return info_gain

In [99]:
def find_best_split(X, y, feature_names):
    best_gain = -1
    best_split = {}

    # 1. Loop through ALL features (Correct for accuracy)
    for feat_idx in range(len(feature_names)):
        current_data = X[:, feat_idx]
        current_name = feature_names[feat_idx]

        # Check type
        if current_name in categorical_cols:
            # Categorical: Check all unique values (usually small, like 0 or 1)
            unique_vals = np.unique(current_data)
            for val in unique_vals:
                gain = calculate_categorical_gain(current_data, y, val)
                if gain > best_gain:
                    best_gain = gain
                    best_split = {'feature_idx': feat_idx, 'threshold': val, 'type': 'categorical'}

        else:
            # Numerical: This is where we must be careful!
            unique_vals = np.unique(current_data)

            # 2. KEY SAFETY STEP: Only check 20-50 random thresholds
            # If we check ALL 10,000 unique lengths, this will freeze.
            if len(unique_vals) > 50:
                check_vals = np.random.choice(unique_vals, 50, replace=False)
            else:
                check_vals = unique_vals

            for val in check_vals:
                gain = calculate_numerical_gain(current_data, y, val)
                if gain > best_gain:
                    best_gain = gain
                    best_split = {'feature_idx': feat_idx, 'threshold': val, 'type': 'numerical'}

    return best_split, best_gain

In [100]:
def build_tree(X, y, depth, max_depth, feature_names):
    # Stop if max depth reached or only 1 class left
    unique_labels = np.unique(y)
    if len(unique_labels) == 1 or depth >= max_depth:
        # Return leaf
        vals, counts = np.unique(y, return_counts=True)
        majority = vals[np.argmax(counts)]
        return {'type': 'leaf', 'value': majority}

    # Find best split
    split_info, gain = find_best_split(X, y, feature_names)

    if gain <= 0:
        vals, counts = np.unique(y, return_counts=True)
        majority = vals[np.argmax(counts)]
        return {'type': 'leaf', 'value': majority}

    # Perform the split based on type
    feat_idx = split_info['feature_idx']
    thresh = split_info['threshold']

    if split_info['type'] == 'categorical':
        left_mask = X[:, feat_idx] == thresh
        right_mask = X[:, feat_idx] != thresh
    else:
        left_mask = X[:, feat_idx] <= thresh
        right_mask = X[:, feat_idx] > thresh

    left_branch = build_tree(X[left_mask], y[left_mask], depth+1, max_depth, feature_names)
    right_branch = build_tree(X[right_mask], y[right_mask], depth+1, max_depth, feature_names)

    return {
        'type': 'node',
        'feature_idx': feat_idx,
        'threshold': thresh,
        'split_type': split_info['type'],
        'left': left_branch,
        'right': right_branch
    }

In [101]:
def predict_row(row, node):
    if node['type'] == 'leaf':
        return node['value']

    # Check split condition
    val = row[node['feature_idx']]

    if node['split_type'] == 'categorical':
        if val == node['threshold']:
            return predict_row(row, node['left'])
        else:
            return predict_row(row, node['right'])
    else:
        # Numerical
        if val <= node['threshold']:
            return predict_row(row, node['left'])
        else:
            return predict_row(row, node['right'])

In [102]:
print("Building Tree (Depth 8)...")
# Note: Max depth 8 and min_samples logic is hidden inside recursion stops
my_tree = build_tree(X_train_arr, y_train_arr, 0, 8, col_names)
print("Tree Built!")

print("Predicting on Test Set...")
predictions = []
for row in X_test_arr:
    pred = predict_row(row, my_tree)
    predictions.append(pred)

# Calculate Accuracy
acc = accuracy_score(y_test, predictions)
print(f"Accuracy: {acc*100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=['good', 'bad']))

Building Tree (Depth 8)...
Tree Built!
Predicting on Test Set...
Accuracy: 98.70%

Classification Report:
              precision    recall  f1-score   support

        good       0.98      1.00      0.99       482
         bad       1.00      0.98      0.99       518

    accuracy                           0.99      1000
   macro avg       0.99      0.99      0.99      1000
weighted avg       0.99      0.99      0.99      1000

